# BirdCLEF+ 2026 Perch v2 Probe

Extract **frozen Google Perch v2 embeddings** and train a shallow PyTorch probe for the 206 BirdCLEF+ 2026 primary labels. Use **`CFG.mode = "train"`** for the full experiment and **`CFG.mode = "submission"`** to load the uploaded probe artifact and score test soundscapes.


## 1. Environment

Configure the Kaggle runtime and load the libraries used for Perch embedding extraction and PyTorch probe training.


In [1]:
from pathlib import Path
import json
import os
import random
import re
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    """Base configuration for the Perch v2 probe notebook."""

    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")
    tensorflow_wheel_root = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0")
    perch_input_roots = [
        Path(
            "/kaggle/input/models/google/"
            "bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1"
        ),
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        Path(
            "/kaggle/input/models/google/"
            "bird-vocalization-classifier/tensorflow2/perch_v2/2"
        ),
        Path("/kaggle/input/perch-meta"),
    ]
    perch_artifact_roots = [
        Path(
            "/kaggle/input/models/tuannm3812/"
            "birdclef-perch-v2-artifacts/pytorch/default/1/perch_v2"
        ),
    ]


def seed_everything(seed: int = 42) -> None:
    """Set deterministic seeds for Python and NumPy.

    Args:
        seed (int): Seed used for reproducible notebook operations.
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    """Find the attached BirdCLEF+ 2026 Kaggle dataset root.

    Returns:
        Path: Directory containing `train.csv`.

    Raises:
        FileNotFoundError: If no candidate dataset directory is found.
    """
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def package_version(package_name: str) -> str | None:
    """Return an installed package version when available.

    Args:
        package_name (str): Distribution name.

    Returns:
        str | None: Installed version string or `None`.
    """
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def version_tuple(value: str) -> tuple[int, ...]:
    """Convert a version string into comparable integer components.

    Args:
        value (str): Version string such as `2.20.0`.

    Returns:
        tuple[int, ...]: Numeric version prefix.
    """
    core = value.split("+")[0].split("-")[0]
    parts = []
    for part in core.split("."):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def install_tensorflow_220() -> None:
    """Install TensorFlow 2.20 from the attached Kaggle wheel input.

    Raises:
        FileNotFoundError: If the TensorFlow wheel cannot be found.
    """
    tf_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorflow-2.20.0*.whl"))
    tb_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorboard-2.20.0*.whl"))
    if not tf_wheels:
        raise FileNotFoundError(
            f"TensorFlow 2.20 wheel not found under {CFG.tensorflow_wheel_root}. "
            "Attach the bc26-tensorflow-2-20-0 Kaggle input before running notebook 3."
        )
    targets = [str(path) for path in tb_wheels[:1] + tf_wheels[:1]]
    print(f"Installing TensorFlow 2.20 from {CFG.tensorflow_wheel_root}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *targets], check=True)


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Output directory: {CFG.artifact_dir}")

import librosa
import soundfile as sf
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

installed_tf = package_version("tensorflow")
if installed_tf is None or version_tuple(installed_tf) < version_tuple("2.20.0"):
    print(
        f"TensorFlow {installed_tf} is older than the Perch v2 requirement; "
        "installing 2.20.0 from Kaggle input."
    )
    install_tensorflow_220()

try:
    import tensorflow as tf
except ImportError:
    tf = None

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)


class CFG(CFG):
    """Perch-specific configuration values."""

    mode = "train"
    artifact_dir = Path("/kaggle/working/artifacts/perch_v2")
    submission_artifact_dir = None
    sample_rate = 32000
    duration = 5.0
    embedding_dim = 1536
    extraction_batch_size = 8
    probe_batch_size = 128
    probe_epochs = 20
    early_stopping_patience = 4
    early_stopping_min_delta = 1e-4
    hidden_dim = 512
    dropout = 0.25
    lr = 1e-3
    max_samples = None
    perch_model_dir = None
    reuse_attached_artifacts = True
    inference_batch_size = 8
    submission_batch_files = 16
    soundscape_seconds = 60
    use_soundscape_priors = True
    prior_artifact_dir = None
    temperature_grid = np.linspace(0.6, 2.5, 39)
    hour_prior_weight = 0.20
    site_prior_weight = 0.10
    cooccurrence_prior_weight = 0.05
    cooccurrence_min_prob = 0.20
    prior_clip = 1.5
    prior_smoothing = 0.5


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Mode: {CFG.mode}")
if tf is not None:
    print(f"TensorFlow: {tf.__version__}")


Data root: /kaggle/input/competitions/birdclef-2026
Output directory: /kaggle/working/artifacts
TensorFlow 2.19.0 is older than the Perch v2 requirement; installing 2.20.0 from Kaggle input.
Installing TensorFlow 2.20 from /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0
Device: cuda
Mode: train
TensorFlow: 2.20.0


## 2. Metadata


Use the same **206 primary labels** as the EfficientNet baseline. Training mode writes the label map; submission mode reads the label order from the Perch probe artifact.


In [2]:
train = pd.read_csv(CFG.data_root / "train.csv")
taxonomy_path = CFG.data_root / "taxonomy.csv"
taxonomy = pd.read_csv(taxonomy_path) if taxonomy_path.exists() else pd.DataFrame()

train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
if CFG.mode == "train" and CFG.max_samples:
    train = train.sample(CFG.max_samples, random_state=CFG.seed).reset_index(drop=True)

labels = sorted(train["primary_label"].astype(str).unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["primary_label"] = train["primary_label"].astype(str)
train["target"] = train["primary_label"].map(label_to_idx)

label_metadata = (
    train.groupby("primary_label")
    .agg(train_support=("filename", "count"))
    .reset_index()
)
if "class_name" in train.columns:
    class_meta = train.groupby("primary_label")["class_name"].first().reset_index()
    label_metadata = label_metadata.merge(class_meta, on="primary_label", how="left")
if not taxonomy.empty and {"primary_label", "class_name"}.issubset(taxonomy.columns):
    tax_meta = taxonomy[["primary_label", "class_name"]].drop_duplicates()
    tax_meta["primary_label"] = tax_meta["primary_label"].astype(str)
    label_metadata = label_metadata.merge(
        tax_meta,
        on="primary_label",
        how="left",
        suffixes=("", "_taxonomy"),
    )
    if "class_name" not in label_metadata.columns:
        label_metadata["class_name"] = label_metadata["class_name_taxonomy"]
    else:
        label_metadata["class_name"] = label_metadata["class_name"].fillna(
            label_metadata["class_name_taxonomy"]
        )
    label_metadata = label_metadata.drop(columns=["class_name_taxonomy"], errors="ignore")

if CFG.mode == "train":
    (CFG.artifact_dir / "labels.json").write_text(
        json.dumps(idx_to_label, indent=2),
        encoding="utf-8",
    )
    label_metadata.to_csv(CFG.artifact_dir / "label_metadata.csv", index=False)

print(f"Rows: {len(train):,}")
print(f"Classes: {len(labels):,}")
display(train.head())


Rows: 35,549
Classes: 206


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection,filepath,target
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0


## 3. Load Perch v2


Load the **Perch v2 TensorFlow SavedModel** and run a one-batch smoke test. The notebook prefers the CPU-specific Perch export for submission because it avoids GPU/XLA overhead and is fast enough for batched soundscape scoring.


In [3]:
def find_perch_model_dir() -> Path:
    """Find the attached Perch TensorFlow SavedModel directory.

    Returns:
        Path: Directory containing `saved_model.pb`.

    Raises:
        FileNotFoundError: If no Perch SavedModel is attached.
    """
    if CFG.perch_model_dir:
        path = Path(CFG.perch_model_dir)
        if (path / "saved_model.pb").exists():
            return path
        matches = list(path.glob("**/saved_model.pb")) if path.exists() else []
        if matches:
            return matches[0].parent

    search_roots = [path for path in CFG.perch_input_roots if path.exists()]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        search_roots.append(input_root)

    matches = []
    for root in dict.fromkeys(search_roots):
        if (root / "saved_model.pb").exists():
            matches.append(root / "saved_model.pb")
        matches.extend(root.glob("**/saved_model.pb"))
    matches = [path.parent for path in matches if "perch" in str(path).lower() or "vocal" in str(path).lower() or "bird" in str(path).lower()]
    if matches:
        return matches[0]
    raise FileNotFoundError(
        "Could not find a Perch SavedModel. Attach /kaggle/input/datasets/jaejohn/perch-meta, "
        "the Perch/vocalization-classifier Kaggle model, or set CFG.perch_model_dir."
    )


if tf is None:
    raise ImportError("TensorFlow is required for Perch embedding extraction.")

perch_model_dir = find_perch_model_dir()
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Perch model: {perch_model_dir}")
print(f"Inputs: {infer.structured_input_signature}")
print(f"Outputs: {infer.structured_outputs}")


def explain_perch_runtime_error(error: Exception) -> None:
    """Raise a clearer message for known Perch runtime failures.

    Args:
        error (Exception): Original TensorFlow exception.

    Raises:
        RuntimeError: For known TensorFlow/XLA incompatibility errors.
        Exception: Re-raises the original error otherwise.
    """
    message = str(error)
    if "XlaCallModuleOp with version 10 is not supported" in message:
        raise RuntimeError(
            "The Perch SavedModel requires a newer TensorFlow/XLA runtime than this Kaggle image. "
            "Use the CPU Perch export or a Kaggle runtime with tensorflow>=2.20."
        ) from error
    raise error


def smoke_test_perch() -> None:
    """Run a one-batch inference pass through the Perch signature."""
    dummy = tf.zeros((1, int(CFG.sample_rate * CFG.duration)), dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    try:
        if keyword_specs:
            input_name = next(iter(keyword_specs))
            outputs = infer(**{input_name: dummy})
        else:
            outputs = infer(dummy)
    except Exception as error:
        explain_perch_runtime_error(error)
    print({name: tuple(value.shape) for name, value in outputs.items()})


smoke_test_perch()


I0000 00:00:1780027258.343766      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780027258.346795      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Perch model: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
Inputs: ((), {'inputs': TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')})
Outputs: {'embedding': TensorSpec(shape=(None, 1536), dtype=tf.float32, name='embedding'), 'spatial_embedding': TensorSpec(shape=(None, 16, 4, 1536), dtype=tf.float32, name='spatial_embedding'), 'spectrogram': TensorSpec(shape=(None, 500, 128), dtype=tf.float32, name='spectrogram'), 'label': TensorSpec(shape=(None, 14795), dtype=tf.float32, name='label')}


2026-05-29 04:01:08.951447: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 04:01:09.092908: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 04:01:10.444556: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 04:01:10.583175: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-29 04:01:10.770043: E external/local_xla/xla/stream_

{'embedding': (1, 1536), 'spatial_embedding': (1, 16, 4, 1536), 'spectrogram': (1, 500, 128), 'label': (1, 14795)}


## 4. Embedding Extraction


Convert each **5-second waveform** into a **1,536-dimensional Perch embedding**. Training mode can reuse cached train embeddings; submission mode extracts only the hidden-test windows needed for `submission.csv`.


In [4]:
def load_audio(path: Path) -> np.ndarray:
    """Load a fixed-duration mono waveform for Perch inference.

    Args:
        path (Path): Audio file path.

    Returns:
        np.ndarray: Mono waveform padded or trimmed to `CFG.duration`.
    """
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(
        path,
        sr=CFG.sample_rate,
        mono=True,
        duration=CFG.duration,
    )
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def load_audio_segment(path: Path, end_time: float) -> np.ndarray:
    """Load one fixed-duration audio segment ending at `end_time`.

    Args:
        path (Path): Audio file path.
        end_time (float): Segment end time in seconds.

    Returns:
        np.ndarray: Mono waveform segment.
    """
    offset = max(0.0, float(end_time) - CFG.duration)
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(
        path,
        sr=CFG.sample_rate,
        mono=True,
        offset=offset,
        duration=CFG.duration,
    )
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def find_perch_artifact_dir() -> Path | None:
    """Find the attached Perch artifact directory when available.

    Returns:
        Path | None: Directory containing `best_perch_probe.pt` and
        `labels.json`, or `None` when no artifact is attached.
    """
    required_files = ["best_perch_probe.pt", "labels.json"]
    candidates = []

    if CFG.submission_artifact_dir is not None:
        candidates.append(Path(CFG.submission_artifact_dir))
    candidates.extend(CFG.perch_artifact_roots)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(path.parent for path in input_root.rglob("best_perch_probe.pt"))

    seen = set()
    for root in candidates:
        root = Path(root)
        if root in seen or not root.exists():
            continue
        seen.add(root)
        if all((root / name).exists() for name in required_files):
            return root

    return None

def load_embedding_cache(path: Path) -> np.ndarray:
    """Load cached Perch embeddings from an `.npz` file.

    Args:
        path (Path): Embedding cache path.

    Returns:
        np.ndarray: Cached embedding matrix.
    """
    saved = np.load(path, allow_pickle=True)
    return saved["embeddings"].astype(np.float32)


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    """Extract Perch embeddings for a batch of waveforms.

    Args:
        batch_waveforms (np.ndarray): Batch shaped `(batch, samples)`.

    Returns:
        np.ndarray: Batch of 2D Perch embeddings.
    """
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    if keyword_specs:
        input_name = next(iter(keyword_specs))
        try:
            outputs = infer(**{input_name: tensor})
        except Exception as error:
            explain_perch_runtime_error(error)
    else:
        try:
            outputs = infer(tensor)
        except Exception as error:
            explain_perch_runtime_error(error)
    arrays = {name: np.asarray(value) for name, value in outputs.items()}
    embedding_name = next(
        (
            name
            for name in arrays
            if "embedding" in name.lower() or "embed" in name.lower()
        ),
        next(iter(arrays)),
    )
    value = arrays[embedding_name]
    if value.ndim == 3:
        value = value.mean(axis=1)
    elif value.ndim > 3:
        value = value.reshape(value.shape[0], -1)
    return value.astype(np.float32)


attached_artifact_dir = find_perch_artifact_dir()
print(f"Attached artifact directory: {attached_artifact_dir}")

if CFG.mode == "train":
    embeddings_path = CFG.artifact_dir / "train_embeddings.npz"
    attached_embeddings_path = None
    if attached_artifact_dir is not None:
        attached_embeddings_path = attached_artifact_dir / "train_embeddings.npz"

    if embeddings_path.exists():
        embeddings = load_embedding_cache(embeddings_path)
        print(f"Loaded embeddings from {embeddings_path}")
    elif CFG.reuse_attached_artifacts and attached_embeddings_path is not None:
        embeddings = load_embedding_cache(attached_embeddings_path)
        print(f"Loaded embeddings from {attached_embeddings_path}")
    else:
        chunks = []
        waveforms = []
        for path in tqdm(train["filepath"], desc="audio"):
            waveforms.append(load_audio(path))
            if len(waveforms) == CFG.extraction_batch_size:
                chunks.append(run_perch_batch(np.stack(waveforms)))
                waveforms = []
        if waveforms:
            chunks.append(run_perch_batch(np.stack(waveforms)))
        if not chunks:
            raise RuntimeError("No training embeddings were generated.")
        embeddings = np.concatenate(chunks, axis=0)
        np.savez_compressed(
            embeddings_path,
            embeddings=embeddings,
            labels=train["primary_label"].to_numpy(),
            filenames=train["filename"].to_numpy(),
        )
        print(f"Saved embeddings to {embeddings_path}")

    print(f"Embeddings: {embeddings.shape}")
else:
    print("Submission mode: skipped train embedding extraction.")


Attached artifact directory: /kaggle/input/models/tuannm3812/birdclef-perch-v2-artifacts/pytorch/default/1/perch_v2
Loaded embeddings from /kaggle/input/models/tuannm3812/birdclef-perch-v2-artifacts/pytorch/default/1/perch_v2/train_embeddings.npz
Embeddings: (35549, 1536)


## 5. Probe Model


The probe is deliberately shallow: **LayerNorm -> Linear -> ReLU -> Dropout -> Linear**. Perch already encodes much of the acoustic structure, so the classifier focuses on compact class boundaries over frozen features.


In [5]:
class PerchProbe(nn.Module):
    """Shallow classifier trained on frozen Perch embeddings."""

    def __init__(self, embedding_dim: int, num_classes: int) -> None:
        """Initialize the probe network.

        Args:
            embedding_dim (int): Width of the Perch embedding vector.
            num_classes (int): Number of target classes.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute class logits for a batch of embeddings.

        Args:
            x (torch.Tensor): Embedding tensor.

        Returns:
            torch.Tensor: Class logits.
        """
        return self.net(x)


def torch_load_checkpoint(path: Path) -> dict[str, object]:
    """Load a PyTorch checkpoint with Kaggle-version compatibility.

    Args:
        path (Path): Checkpoint path.

    Returns:
        dict[str, object]: Loaded checkpoint dictionary.
    """
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_artifact_labels(artifact_dir: Path) -> list[str]:
    """Load label order from an attached Perch artifact.

    Args:
        artifact_dir (Path): Directory containing `labels.json`.

    Returns:
        list[str]: Labels ordered by model output index.
    """
    labels_path = artifact_dir / "labels.json"
    idx_to_label_raw = json.loads(labels_path.read_text(encoding="utf-8"))
    return [idx_to_label_raw[str(i)] for i in range(len(idx_to_label_raw))]


if CFG.mode == "train":
    x = embeddings.astype(np.float32)
    y = train["target"].to_numpy(dtype=np.int64)

    def safe_train_valid_split(
        targets: np.ndarray,
        test_size: float = 0.2,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Create a stratified split while keeping singleton classes in train.

        Args:
            targets (np.ndarray): Integer class targets.
            test_size (float): Validation fraction for non-singleton classes.

        Returns:
            tuple[np.ndarray, np.ndarray]: Train and validation row indices.
        """
        counts = pd.Series(targets).value_counts()
        rare_classes = set(counts[counts < 2].index)
        all_idx = np.arange(len(targets))
        rare_idx = np.array(
            [idx for idx in all_idx if targets[idx] in rare_classes],
            dtype=np.int64,
        )
        common_idx = np.array(
            [idx for idx in all_idx if targets[idx] not in rare_classes],
            dtype=np.int64,
        )

        train_common, valid_idx = train_test_split(
            common_idx,
            test_size=test_size,
            random_state=CFG.seed,
            stratify=targets[common_idx],
        )
        train_idx = np.concatenate([train_common, rare_idx])
        return train_idx, valid_idx

    train_idx, valid_idx = safe_train_valid_split(y)
    valid_meta = train.iloc[valid_idx][
        ["filename", "primary_label", "target"]
    ].reset_index(drop=True)
    print(f"Probe train rows: {len(train_idx):,}")
    print(f"Probe valid rows: {len(valid_idx):,}")
    rare_count = int((pd.Series(y).value_counts() < 2).sum())
    print(f"Classes with fewer than 2 rows kept in train only: {rare_count:,}")

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x[train_idx]), torch.from_numpy(y[train_idx])),
        batch_size=CFG.probe_batch_size,
        shuffle=True,
    )
    valid_loader = DataLoader(
        TensorDataset(torch.from_numpy(x[valid_idx]), torch.from_numpy(y[valid_idx])),
        batch_size=CFG.probe_batch_size * 2,
        shuffle=False,
    )

    model = PerchProbe(embedding_dim=x.shape[1], num_classes=len(labels)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
else:
    if attached_artifact_dir is None:
        raise FileNotFoundError(
            "Submission mode requires an attached Perch artifact directory."
        )
    labels = load_artifact_labels(attached_artifact_dir)
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    idx_to_label = {idx: label for label, idx in label_to_idx.items()}
    checkpoint_path = attached_artifact_dir / "best_perch_probe.pt"
    checkpoint = torch_load_checkpoint(checkpoint_path)
    model = PerchProbe(
        embedding_dim=CFG.embedding_dim,
        num_classes=len(labels),
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    print(f"Loaded probe checkpoint: {checkpoint_path}")
    print(f"Loaded labels: {len(labels):,}")


Probe train rows: 28,440
Probe valid rows: 7,109
Classes with fewer than 2 rows kept in train only: 4


## 6. Training


Rare singleton classes stay in training so the stratified validation split remains valid. The training loop tracks **best validation accuracy**, applies **early stopping**, and writes diagnostic files for per-class analysis.


In [6]:
if CFG.mode == "train":
    @torch.no_grad()
    def evaluate(loader: DataLoader) -> tuple[float, float]:
        """Evaluate probe loss and top-1 accuracy."""
        model.eval()
        total_loss = 0.0
        correct = 0
        seen = 0
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += xb.size(0)
        return total_loss / max(seen, 1), correct / max(seen, 1)


    @torch.no_grad()
    def collect_validation_logits() -> tuple[np.ndarray, np.ndarray]:
        """Collect validation logits and target indices from the best probe."""
        model.eval()
        logits_chunks, target_chunks = [], []
        for xb, yb in valid_loader:
            xb = xb.to(device)
            logits_chunks.append(model(xb).cpu().numpy())
            target_chunks.append(yb.numpy())
        return np.concatenate(logits_chunks), np.concatenate(target_chunks)


    def softmax_numpy(logits: np.ndarray) -> np.ndarray:
        """Compute a stable NumPy softmax."""
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(shifted)
        return exp / exp.sum(axis=1, keepdims=True)


    def negative_log_loss(logits: np.ndarray, targets: np.ndarray) -> float:
        """Compute multiclass negative log loss from logits."""
        shifted = logits - logits.max(axis=1, keepdims=True)
        log_probs = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        return float(-log_probs[np.arange(len(targets)), targets].mean())


    def fit_temperature(logits: np.ndarray, targets: np.ndarray) -> dict[str, float]:
        """Tune a global temperature on validation logits."""
        rows = []
        for temp in CFG.temperature_grid:
            rows.append(
                {
                    "temperature": float(temp),
                    "valid_log_loss": negative_log_loss(logits / float(temp), targets),
                }
            )
        calibration_df = pd.DataFrame(rows).sort_values("valid_log_loss")
        calibration_df.to_csv(CFG.artifact_dir / "temperature_grid.csv", index=False)
        best = calibration_df.iloc[0].to_dict()
        return {
            "temperature": float(best["temperature"]),
            "valid_log_loss": float(best["valid_log_loss"]),
            "uncalibrated_valid_log_loss": negative_log_loss(logits, targets),
        }


    def collect_validation_predictions_from_logits(
        logits: np.ndarray,
        targets: np.ndarray,
    ) -> tuple[pd.DataFrame, pd.DataFrame, float]:
        """Create row-level and per-class validation diagnostics."""
        probs = softmax_numpy(logits)
        k = min(5, probs.shape[1])
        top_indices = np.argsort(-probs, axis=1)[:, :k]
        top_values = np.take_along_axis(probs, top_indices, axis=1)

        pred_df = valid_meta.copy()
        pred_df["top1_label"] = [idx_to_label[int(idx)] for idx in top_indices[:, 0]]
        pred_df["top1_prob"] = top_values[:, 0]
        pred_df["top5_labels"] = [
            " ".join(idx_to_label[int(idx)] for idx in row) for row in top_indices
        ]
        pred_df["top1_correct"] = pred_df["primary_label"].eq(pred_df["top1_label"]).astype(int)
        pred_df["top5_correct"] = [
            int(int(target) in row) for target, row in zip(targets, top_indices)
        ]

        class_df = (
            pred_df.groupby("primary_label")
            .agg(
                support=("filename", "count"),
                top1_recall=("top1_correct", "mean"),
                top5_recall=("top5_correct", "mean"),
                mean_top1_prob=("top1_prob", "mean"),
            )
            .reset_index()
        )
        class_df = class_df.merge(label_metadata, on="primary_label", how="left")
        class_df = class_df.sort_values(
            ["top1_recall", "top5_recall", "support"],
            ascending=[True, True, False],
        )
        return pred_df, class_df, float(pred_df["top5_correct"].mean())


    def summarize_weak_labels(
        pred_df: pd.DataFrame,
        class_df: pd.DataFrame,
    ) -> pd.DataFrame:
        """Rank labels that need calibration, confusion, or data work."""
        error_df = pred_df[pred_df["top1_correct"].eq(0)]
        error_summary = (
            error_df.groupby("primary_label")
            .agg(
                high_conf_errors=("top1_prob", lambda s: int((s >= 0.50).sum())),
                mean_error_confidence=("top1_prob", "mean"),
                most_common_wrong_label=("top1_label", lambda s: s.value_counts().index[0]),
            )
            .reset_index()
        ) if len(error_df) else pd.DataFrame(columns=["primary_label"])
        weak = class_df.merge(error_summary, on="primary_label", how="left")
        weak["high_conf_errors"] = weak["high_conf_errors"].fillna(0).astype(int)
        weak["mean_error_confidence"] = weak["mean_error_confidence"].fillna(0.0)
        weak["most_common_wrong_label"] = weak["most_common_wrong_label"].fillna("")
        weak["weak_label_score"] = (
            (1.0 - weak["top1_recall"])
            + 0.5 * (1.0 - weak["top5_recall"])
            + 0.25 * weak["mean_error_confidence"]
        )
        return weak.sort_values(
            ["weak_label_score", "support"],
            ascending=[False, False],
        )


    history = []
    best_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(1, CFG.probe_epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)

        valid_loss, valid_acc = evaluate(valid_loader)
        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(len(train_loader.dataset), 1),
            "valid_loss": valid_loss,
            "valid_acc": valid_acc,
        }
        history.append(row)
        print(row)

        improved = valid_acc > best_acc + CFG.early_stopping_min_delta
        if improved:
            best_acc = valid_acc
            epochs_without_improvement = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "label_to_idx": label_to_idx,
                    "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                    "valid_acc": best_acc,
                },
                CFG.artifact_dir / "best_perch_probe.pt",
            )
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= CFG.early_stopping_patience:
                print(f"Early stopping after {epoch} epochs; best valid accuracy: {best_acc:.4f}")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(CFG.artifact_dir / "history.csv", index=False)

    best_checkpoint = torch_load_checkpoint(CFG.artifact_dir / "best_perch_probe.pt")
    model.load_state_dict(best_checkpoint["model"])
    valid_logits, valid_targets = collect_validation_logits()
    validation_predictions, per_class_metrics, valid_top5_acc = collect_validation_predictions_from_logits(
        valid_logits,
        valid_targets,
    )
    calibration = fit_temperature(valid_logits, valid_targets)
    weak_label_diagnostics = summarize_weak_labels(validation_predictions, per_class_metrics)

    validation_predictions.to_csv(CFG.artifact_dir / "validation_predictions.csv", index=False)
    per_class_metrics.to_csv(CFG.artifact_dir / "per_class_validation_metrics.csv", index=False)
    weak_label_diagnostics.to_csv(CFG.artifact_dir / "weak_label_diagnostics.csv", index=False)
    (CFG.artifact_dir / "calibration.json").write_text(
        json.dumps(calibration, indent=2),
        encoding="utf-8",
    )
    summary = {
        "best_valid_acc": float(best_acc),
        "valid_top5_acc": valid_top5_acc,
        "valid_log_loss": negative_log_loss(valid_logits, valid_targets),
        "calibrated_valid_log_loss": calibration["valid_log_loss"],
        "temperature": calibration["temperature"],
        "epochs_ran": len(history_df),
        "best_epoch": int(history_df.loc[history_df["valid_acc"].idxmax(), "epoch"]),
        "embedding_shape": list(embeddings.shape),
    }
    (CFG.artifact_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"Best valid accuracy: {best_acc:.4f}")
    print(f"Validation top-5 accuracy: {valid_top5_acc:.4f}")
    print(
        "Temperature calibration: "
        f"T={calibration['temperature']:.3f}, "
        f"log loss {calibration['uncalibrated_valid_log_loss']:.4f} -> "
        f"{calibration['valid_log_loss']:.4f}"
    )
    print(f"Saved outputs to {CFG.artifact_dir}")
    display(weak_label_diagnostics.head(15))
else:
    print("Submission mode: skipped probe training and diagnostics.")


{'epoch': 1, 'train_loss': 1.3073418041740958, 'valid_loss': 0.7493900626516322, 'valid_acc': 0.8357012238008159}
{'epoch': 2, 'train_loss': 0.5470918343917227, 'valid_loss': 0.735564296214385, 'valid_acc': 0.8333098888732593}
{'epoch': 3, 'train_loss': 0.33816701929109844, 'valid_loss': 0.7523926093468, 'valid_acc': 0.839077226051484}
{'epoch': 4, 'train_loss': 0.21085617468494572, 'valid_loss': 0.7947470974798091, 'valid_acc': 0.8366858911239274}
{'epoch': 5, 'train_loss': 0.1426572760491096, 'valid_loss': 0.8367681127758968, 'valid_acc': 0.8347165564777043}
{'epoch': 6, 'train_loss': 0.11112977336730635, 'valid_loss': 0.8886440724222481, 'valid_acc': 0.8321845547897032}
{'epoch': 7, 'train_loss': 0.09095499820315636, 'valid_loss': 0.9148387177900383, 'valid_acc': 0.8348572232381488}
Early stopping after 7 epochs; best valid accuracy: 0.8391
Best valid accuracy: 0.8391
Validation top-5 accuracy: 0.9278
Temperature calibration: T=1.200, log loss 0.7524 -> 0.7264
Saved outputs to /kagg

,primary_label,support,top1_recall,top5_recall,mean_top1_prob,train_support,class_name,high_conf_errors,mean_error_confidence,most_common_wrong_label,weak_label_score
2,22961,1,0.000000,0.000000,0.720152,6,Amphibia,1,0.720152,65380,1.680038
1,22930,1,0.000000,0.000000,0.636764,6,Amphibia,1,0.636764,squcuc1,1.659191
5,67252,1,0.000000,0.000000,0.575881,6,Amphibia,1,0.575881,roahaw,1.643970
4,555123,1,0.000000,0.000000,0.241889,3,Amphibia,0,0.241889,trsowl,1.560472
0,1595929,1,0.000000,0.000000,0.221081,5,Amphibia,0,0.221081,watjac1,1.555270
3,476521,1,0.000000,0.000000,0.111824,3,Amphibia,0,0.111824,coffal1,1.527956
7,555145,2,0.000000,0.500000,0.882033,9,Amphibia,2,0.882033,compot1,1.470508
6,22983,2,0.000000,0.500000,0.598941,10,Amphibia,1,0.598941,65377,1.399735
8,74113,2,0.000000,0.500000,0.530428,10,Mammalia,1,0.530428,soulap1,1.382607
9,23154,1,0.000000,1.000000,0.699673,5,Amphibia,1,0.699673,326272,1.174918


## 7. Soundscape Priors And Calibration

Build lightweight priors from deduplicated labeled soundscape windows. These are saved as artifacts in training mode and can be loaded in submission mode when `CFG.use_soundscape_priors = True`. The defaults keep priors disabled for score comparability until a Kaggle run verifies the effect.

In [7]:
def split_soundscape_labels(value: object) -> list[str]:
    """Split semicolon-delimited soundscape labels."""
    if pd.isna(value):
        return []
    return [label.strip() for label in str(value).split(";") if label.strip()]


def parse_soundscape_context(value: object) -> dict[str, object]:
    """Extract site and hour from a soundscape filename or row id."""
    stem = Path(str(value)).stem
    stem = re.sub(r"_\d+(?:\.0)?$", "", stem)
    parts = stem.split("_")
    site = next((part for part in parts if re.fullmatch(r"S\d+", part)), "unknown")
    time_part = next((part for part in reversed(parts) if re.fullmatch(r"\d{6}", part)), None)
    hour = int(time_part[:2]) if time_part else -1
    return {"site": site, "hour": hour}


def logit(values: np.ndarray) -> np.ndarray:
    """Convert probabilities to logits with clipping."""
    values = np.clip(values, 1e-5, 1 - 1e-5)
    return np.log(values / (1 - values))


def build_prior_table(
    expanded: pd.DataFrame,
    context_col: str,
    global_probs: np.ndarray,
) -> dict[str, list[float]]:
    """Build smoothed context-specific logit offsets."""
    table = {}
    for context, group in expanded.groupby(context_col):
        counts = group["label"].value_counts()
        total = max(group["segment_key"].nunique(), 1)
        probs = np.full(len(labels), CFG.prior_smoothing, dtype=np.float32)
        for label, count in counts.items():
            if label in label_to_idx:
                probs[label_to_idx[label]] += float(count)
        probs = probs / (total + 2 * CFG.prior_smoothing)
        offsets = np.clip(logit(probs) - logit(global_probs), -CFG.prior_clip, CFG.prior_clip)
        table[str(context)] = offsets.astype(float).tolist()
    return table


def build_soundscape_prior_artifact() -> dict[str, object] | None:
    """Create prior tables from train_soundscapes_labels.csv when available."""
    path = CFG.data_root / "train_soundscapes_labels.csv"
    if not path.exists():
        print("No train_soundscapes_labels.csv found; priors disabled.")
        return None

    sc = pd.read_csv(path).drop_duplicates().reset_index(drop=True)
    if not {"filename", "primary_label"}.issubset(sc.columns):
        print("Soundscape label schema is missing filename/primary_label; priors disabled.")
        return None

    context = sc["filename"].map(parse_soundscape_context).apply(pd.Series)
    sc = pd.concat([sc, context], axis=1)
    if "row_id" in sc.columns:
        sc["segment_key"] = sc["row_id"].astype(str)
    else:
        sc["segment_key"] = sc["filename"].astype(str)
        for time_col in ["seconds", "end_time", "end_seconds"]:
            if time_col in sc.columns:
                sc["segment_key"] = sc["segment_key"] + "_" + sc[time_col].astype(str)
                break
    sc["label_list"] = sc["primary_label"].map(split_soundscape_labels)
    expanded = sc.explode("label_list").rename(columns={"label_list": "label"})
    expanded = expanded[expanded["label"].isin(label_to_idx)].copy()
    if expanded.empty:
        print("No soundscape labels overlap the model label set; priors disabled.")
        return None

    segment_count = max(sc["segment_key"].nunique(), 1)
    global_counts = expanded["label"].value_counts()
    global_probs = np.full(len(labels), CFG.prior_smoothing, dtype=np.float32)
    for label, count in global_counts.items():
        global_probs[label_to_idx[label]] += float(count)
    global_probs = global_probs / (segment_count + 2 * CFG.prior_smoothing)

    pair_counts = {}
    for label_list in sc["label_list"]:
        clean = sorted({label for label in label_list if label in label_to_idx})
        for left, right in combinations(clean, 2):
            pair_counts.setdefault(left, {})[right] = pair_counts.setdefault(left, {}).get(right, 0) + 1
            pair_counts.setdefault(right, {})[left] = pair_counts.setdefault(right, {}).get(left, 0) + 1

    artifact = {
        "labels": labels,
        "global_logit_offset": np.clip(logit(global_probs), -CFG.prior_clip, CFG.prior_clip).astype(float).tolist(),
        "hour_offsets": build_prior_table(expanded, "hour", global_probs),
        "site_offsets": build_prior_table(expanded, "site", global_probs),
        "cooccurrence_counts": pair_counts,
        "summary": {
            "soundscape_rows": int(len(sc)),
            "deduplicated_segments": int(segment_count),
            "expanded_label_rows": int(len(expanded)),
            "overlap_labels": int(expanded["label"].nunique()),
        },
    }
    path = CFG.artifact_dir / "soundscape_priors.json"
    path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")

    summary_df = pd.Series(artifact["summary"]).to_frame("value")
    summary_df.to_csv(CFG.artifact_dir / "soundscape_prior_summary.csv")
    print(f"Saved soundscape priors to {path}")
    display(summary_df)
    return artifact


def find_prior_artifact_dir() -> Path | None:
    """Find a directory containing calibration and prior artifacts."""
    candidates = []
    if CFG.prior_artifact_dir is not None:
        candidates.append(Path(CFG.prior_artifact_dir))
    candidates.append(CFG.artifact_dir)
    if attached_artifact_dir is not None:
        candidates.append(attached_artifact_dir)
    for root in candidates:
        root = Path(root)
        if (root / "calibration.json").exists() or (root / "soundscape_priors.json").exists():
            return root
    return None


def load_json_if_exists(path: Path) -> dict[str, object] | None:
    """Load a JSON artifact when present."""
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


if CFG.mode == "train":
    soundscape_priors = build_soundscape_prior_artifact()
else:
    prior_root = find_prior_artifact_dir()
    calibration = load_json_if_exists(prior_root / "calibration.json") if prior_root else None
    soundscape_priors = load_json_if_exists(prior_root / "soundscape_priors.json") if prior_root else None
    if soundscape_priors and soundscape_priors.get("labels") != labels:
        print("Soundscape prior label order does not match model labels; priors disabled.")
        soundscape_priors = None
    print(f"Prior artifact directory: {prior_root}")
    print(f"Calibration loaded: {calibration is not None}")
    print(f"Soundscape priors loaded: {soundscape_priors is not None}")


Saved soundscape priors to /kaggle/working/artifacts/perch_v2/soundscape_priors.json


,value
soundscape_rows,739
deduplicated_segments,66
expanded_label_rows,2150
overlap_labels,47


## 8. Submission

Load `sample_submission.csv`, extract Perch embeddings for hidden-test windows, score them with the probe checkpoint, and write `/kaggle/working/submission.csv`.


In [8]:
if CFG.mode == "submission":
    def row_to_stem_and_end_time(row_id: str) -> tuple[str, float]:
        """Parse a submission row id into audio stem and window end time.

        Args:
            row_id (str): Submission row id.

        Returns:
            tuple[str, float]: Audio stem and segment end time in seconds.
        """
        stem, sep, end = str(row_id).rpartition("_")
        if sep and end.replace(".", "", 1).isdigit():
            return stem, float(end)
        return str(row_id), CFG.duration


    def build_audio_index() -> dict[str, Path]:
        """Index hidden-test audio files by filename stem.

        Returns:
            dict[str, Path]: Mapping from audio stem to file path.
        """
        audio_index = {}
        for folder in [CFG.data_root / "test_soundscapes", CFG.data_root / "test_audio"]:
            if folder.exists():
                for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                    for path in folder.glob(ext):
                        audio_index[path.stem] = path
        return audio_index


    def read_soundscape(path: Path) -> np.ndarray:
        """Read a full soundscape file and normalize its length.

        Args:
            path (Path): Soundscape audio path.

        Returns:
            np.ndarray: Mono waveform padded or trimmed to `CFG.soundscape_seconds`.
        """
        target_len = int(CFG.sample_rate * CFG.soundscape_seconds)
        y, sr = sf.read(path, dtype="float32", always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr != CFG.sample_rate:
            y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        return y[:target_len].astype(np.float32)


    def infer_soundscape_files(paths: list[Path]) -> tuple[list[str], np.ndarray]:
        """Extract Perch embeddings for all 5-second windows in files.

        Args:
            paths (list[Path]): Soundscape audio paths.

        Returns:
            tuple[list[str], np.ndarray]: Row ids and embeddings.
        """
        row_ids = []
        chunks = []
        windows_per_file = int(CFG.soundscape_seconds / CFG.duration)
        batch_files = int(CFG.submission_batch_files)

        iterator = range(0, len(paths), batch_files)
        iterator = tqdm(
            iterator,
            total=(len(paths) + batch_files - 1) // batch_files,
            desc="perch files",
        )
        for start in iterator:
            batch_paths = paths[start:start + batch_files]
            waveforms = np.empty(
                (len(batch_paths) * windows_per_file, int(CFG.sample_rate * CFG.duration)),
                dtype=np.float32,
            )
            cursor = 0
            for audio_path in batch_paths:
                y = read_soundscape(audio_path)
                windows = y.reshape(windows_per_file, -1)
                waveforms[cursor:cursor + windows_per_file] = windows
                row_ids.extend(
                    f"{audio_path.stem}_{end_time}"
                    for end_time in range(5, CFG.soundscape_seconds + 1, 5)
                )
                cursor += windows_per_file
            chunks.append(run_perch_batch(waveforms))
        return row_ids, np.concatenate(chunks, axis=0)


    def prior_offsets_for_rows(row_ids: list[str], probs: np.ndarray) -> np.ndarray:
        """Build optional row-specific prior offsets for submission logits."""
        offsets = np.zeros((len(row_ids), len(labels)), dtype=np.float32)
        if not CFG.use_soundscape_priors or not soundscape_priors:
            return offsets

        hour_offsets = soundscape_priors.get("hour_offsets", {})
        site_offsets = soundscape_priors.get("site_offsets", {})
        co_counts = soundscape_priors.get("cooccurrence_counts", {})
        for row_idx, row_id in enumerate(row_ids):
            context = parse_soundscape_context(row_id)
            hour_key = str(context["hour"])
            site_key = str(context["site"])
            if hour_key in hour_offsets:
                offsets[row_idx] += CFG.hour_prior_weight * np.asarray(hour_offsets[hour_key], dtype=np.float32)
            if site_key in site_offsets:
                offsets[row_idx] += CFG.site_prior_weight * np.asarray(site_offsets[site_key], dtype=np.float32)

            active = np.flatnonzero(probs[row_idx] >= CFG.cooccurrence_min_prob)
            for active_idx in active:
                active_label = labels[int(active_idx)]
                for linked_label, count in co_counts.get(active_label, {}).items():
                    if linked_label in label_to_idx:
                        boost = np.log1p(float(count)) / 10.0
                        offsets[row_idx, label_to_idx[linked_label]] += CFG.cooccurrence_prior_weight * boost
        return np.clip(offsets, -CFG.prior_clip, CFG.prior_clip)


    @torch.no_grad()
    def predict_embeddings(
        embeddings: np.ndarray,
        row_ids: list[str] | None = None,
    ) -> np.ndarray:
        """Predict class probabilities for Perch embeddings."""
        logits_chunks = []
        for start in range(0, len(embeddings), CFG.probe_batch_size):
            batch = torch.from_numpy(embeddings[start:start + CFG.probe_batch_size]).to(device)
            logits_chunks.append(model(batch).cpu().numpy())
        logits = np.concatenate(logits_chunks, axis=0)

        temperature = 1.0
        if calibration is not None:
            temperature = float(calibration.get("temperature", 1.0))
        logits = logits / max(temperature, 1e-6)
        probs = softmax_numpy(logits) if "softmax_numpy" in globals() else None
        if row_ids is not None and CFG.use_soundscape_priors and soundscape_priors:
            if probs is None:
                shifted = logits - logits.max(axis=1, keepdims=True)
                exp = np.exp(shifted)
                probs = exp / exp.sum(axis=1, keepdims=True)
            logits = logits + prior_offsets_for_rows(row_ids, probs)

        shifted = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(shifted)
        return exp / exp.sum(axis=1, keepdims=True)


    sample_path = CFG.data_root / "sample_submission.csv"
    sample_submission = pd.read_csv(sample_path)
    target_columns = [col for col in sample_submission.columns if col != "row_id"]
    target_positions = [idx for idx, label in enumerate(labels) if label in target_columns]
    target_labels = [labels[idx] for idx in target_positions]
    audio_index = build_audio_index()

    print(f"Sample submission: {sample_path}")
    print(f"Rows: {len(sample_submission):,}")
    print(f"Indexed audio files: {len(audio_index):,}")
    print(f"Submission target columns: {len(target_columns):,}")
    print(f"Model target overlap: {len(target_labels):,}")

    submission = sample_submission.copy()
    for col in target_columns:
        submission[col] = 0.0

    if len(audio_index) == 0 and len(submission) <= 3:
        print("Tiny public dry-run detected with no test audio; preserving sample submission values.")
        submission = sample_submission.copy()
    else:
        test_paths = sorted(audio_index.values())
        row_ids, test_embeddings = infer_soundscape_files(test_paths)
        probs = predict_embeddings(test_embeddings, row_ids)[:, target_positions]
        pred_df = pd.DataFrame(probs, columns=target_labels)
        pred_df.insert(0, "row_id", row_ids)
        pred_df = pred_df.set_index("row_id")

        missing_rows = sorted(set(submission["row_id"]) - set(pred_df.index))
        if missing_rows:
            raise FileNotFoundError(
                f"Missing predictions for {len(missing_rows):,} submission rows. "
                f"First missing row_id={missing_rows[0]}"
            )
        submission[target_labels] = pred_df.loc[submission["row_id"], target_labels].to_numpy()

    submission_path = Path("/kaggle/working/submission.csv")
    submission.to_csv(submission_path, index=False)
    print(f"Wrote submission: {submission_path}")
    display(submission.head())
else:
    print("Train mode: skipped submission inference.")


Train mode: skipped submission inference.


## 9. Interpretation

- **Perch embeddings** provide the strongest validation and public-score signal in this repo. They now serve as the lead submission path and as candidate teacher features for distillation.
- **Early stopping** matters: validation peaks quickly while train loss keeps falling, which suggests the probe can overfit frozen features.
- **Per-class recall** and **top-5 hit rate** identify which labels are already well represented by Perch and which labels still need sampling, augmentation, or domain-specific thresholds.
- Direct Perch inference is operationally heavier than EfficientNet, but the CPU export plus full-file batching has made it viable for scoring.


## 10. Artifacts


Package the **embedding cache**, **best probe checkpoint**, **training history**, **validation predictions**, and **per-class metrics** for review outside the notebook run.


In [9]:
import zipfile
from IPython.display import FileLink

if CFG.mode == "train":
    zip_path = Path("/kaggle/working/birdclef_perch_v2_artifacts.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(CFG.artifact_dir.rglob("*")):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

    print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
    display(FileLink(zip_path))
else:
    print("Submission mode: skipped training artifact zip.")


Wrote /kaggle/working/birdclef_perch_v2_artifacts.zip (3.37 MB)


/kaggle/working/birdclef_perch_v2_artifacts.zip